# MFI: Modern Fortran Interfaces — GPU/CuBLAS Test

**Repo:** https://github.com/14NGiestas/mfi/tree/impl/cublas

Sets up the Fortran + CUDA toolchain on Colab, builds with cuBLAS, and tests:
- Lazy init (no setup code needed)
- `MFI_USE_CUBLAS=1` env var activation
- `mfi_force_gpu` / `mfi_force_cpu` runtime switching
- CPU vs GPU correctness verification
- OpenMP thread safety

**Runtime:** Select **GPU** (T4) before running.

In [ ]:
# 1. Verify GPU is available
!nvidia-smi

In [ ]:
# 2. Diagnose CUDA / cuBLAS availability
import os, subprocess, glob

print("=== CUDA DRIVER ===")
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
for line in result.stdout.split('\n')[:5]:
    print(line)

print("\n=== CUDA TOOLKIT ===")
cuda_candidates = glob.glob("/usr/local/cuda*") + glob.glob("/usr/lib/cuda*")
for p in cuda_candidates:
    print(f"  Found: {p}")

print("\n=== cuBLAS libraries ===")
!ldconfig -p 2>/dev/null | grep cublas || echo "  (not in ldconfig)"
!find /usr -name "libcublas*" 2>/dev/null | head -5

print("\n=== cuBLAS headers ===")
!find /usr -name "cublas*.h" 2>/dev/null | head -5

print("\n=== nvcc ===")
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "  nvcc NOT FOUND")

In [ ]:
# 3. Install Fortran toolchain + BLAS/LAPACK
!apt-get update -qq && apt-get install -y -qq \
    gfortran libblas-dev liblapack-dev libopenblas-dev \
    python3-pip 2>&1 | tail -3
!pip install -q fypp

# Install fpm 0.13.0 (supports profiles)
!curl -sL https://github.com/fortran-lang/fpm/releases/download/v0.13.0/fpm-0.13.0-linux-x86_64-gcc-12 \
    -o /usr/local/bin/fpm && chmod +x /usr/local/bin/fpm

!echo '--- versions ---'
!gfortran --version | head -1
!fpm --version
!fypp --version

In [ ]:
# 4. Install CUDA Toolkit (headers + libraries) if missing
import subprocess, os, glob

r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
has_nvcc = r.returncode == 0

if not has_nvcc:
    print("Installing CUDA toolkit...")
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "nvidia-cuda-toolkit"],
        capture_output=True, text=True
    )
    r2 = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    if r2.returncode != 0:
        print("Trying libcublas-dev...")
        subprocess.run(
            ["apt-get", "install", "-y", "-qq", "libcublas-dev"],
            capture_output=True, text=True
        )
    print("\n=== After install ===")
    !nvcc --version 2>&1 | head -4
else:
    print("CUDA toolkit already installed")
    !nvcc --version | head -4

# Set up paths
header_found = glob.glob("/usr/local/cuda*/include/cuda_runtime.h")
if not header_found:
    header_found = glob.glob("/usr/include/cuda_runtime.h")

if header_found:
    include_dir = os.path.dirname(header_found[0])
    cuda_base = os.path.dirname(include_dir)
    lib_dir = f"{cuda_base}/lib64"
    if not os.path.isdir(lib_dir):
        lib_dir = f"{cuda_base}/targets/x86_64-linux/lib"
    if not os.path.isdir(lib_dir):
        lib_dir = "/usr/lib/x86_64-linux-gnu"

    os.environ["CPATH"] = f"{include_dir}:{os.environ.get('CPATH', '')}"
    os.environ["LIBRARY_PATH"] = f"{lib_dir}:{os.environ.get('LIBRARY_PATH', '')}"
    os.environ["LD_LIBRARY_PATH"] = f"{lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
    print(f"CUDA headers: {include_dir}")
    print(f"CUDA libs:    {lib_dir}")
else:
    print("WARNING: cuda_runtime.h not found")

In [ ]:
# 5. Clone MFI
import subprocess
%cd /content
!rm -rf mfi
!git clone -b impl/cublas --single-branch https://github.com/14NGiestas/mfi.git
%cd mfi
GIT_SHA = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"\nCloned commit: {GIT_SHA}")
!git log -1 --oneline

In [ ]:
# 6. Generate .f90 sources
%cd /content/mfi
!make clean 2>/dev/null
!make
print("\nSource generation complete")

In [ ]:
# 7. Build with cuBLAS
%cd /content/mfi
!fpm build --profile cublas 2>&1

In [ ]:
# 8. Run full test suite with GPU enabled
%cd /content/mfi
import os
os.environ["MFI_USE_CUBLAS"] = "1"
!fpm test --profile cublas 2>&1

In [ ]:
# 9. Quick API verification — compile and run a small test program
%cd /content/mfi

!cat > /tmp/api_test.f90 << 'FORTRAN_EOF'
program api_test
    use mfi_blas, only: mfi_gemm
    use mfi_blas, only: mfi_force_gpu, mfi_force_cpu
    use iso_fortran_env
    implicit none
    real(real64) :: A(4,4), B(4,4), C_cpu(4,4), C_gpu(4,4)
    real(real64) :: diff

    call random_number(A)
    call random_number(B)
    C_cpu = 0.0_real64
    C_gpu = 0.0_real64

    ! CPU (default — lazy init, no setup needed)
    print *, "Testing CPU (default, lazy init)..."
    call mfi_gemm(A, B, C_cpu)

    ! GPU via mfi_force_gpu
    print *, "Testing mfi_force_gpu / mfi_force_cpu..."
    call mfi_force_gpu
    call mfi_gemm(A, B, C_gpu)
    call mfi_force_cpu

    ! Verify results match
    diff = maxval(abs(C_gpu - C_cpu))
    if (diff < 1e-6) then
        print *, "PASS: CPU and GPU results match (diff =", diff, ")"
    else
        print *, "FAIL: CPU/GPU mismatch (diff =", diff, ")"
        error stop
    end if

    print *, "API test complete."
end program
FORTRAN_EOF

import os
os.environ["MFI_USE_CUBLAS"] = "1"

result = !fpm run --profile cublas --file api_test --source /tmp/api_test.f90 2>&1
print("\n".join(result))

In [ ]:
# 10. Test: build WITHOUT cuBLAS — verify mfi_force_gpu/cpu are no-op stubs
%cd /content/mfi
!fpm build 2>&1

# The same source should compile and run on CPU without errors
result = !fpm run --file api_test --source /tmp/api_test.f90 2>&1
print("\n".join(result))